In [0]:
#   Databricks notebook source
"""
Buzzmetrix two-stage topic pipeline for Databricks.

Stage 1
-------
Collect memo samples by category/sentiment group and extract an auto-topic
catalog from the group samples.

Stage 2
-------
Assign each memo to one auto topic from that catalog.

Stage 3
-------
Map the auto topics to a stable taxonomy and save detail / summary tables.

This version is useful when you want to:
- see the themes emerge from the data first
- then map those themes into your business taxonomy
- keep the intermediate auto-topic layer for analysis
"""


from __future__ import annotations

import json
import os
import re
import time
from typing import Any, Dict, Iterator, List, Optional, Tuple

import pandas as pd
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T


# =========================
# Manual Config
# =========================
SNAPSHOT_QUARTER = "2026Q1"
PROMPT_VERSION = "v2stage_1"
ENDPOINT = "databricks-gpt-5-4-mini"
RUN_SCOPE = "PROD"

TEST_MODE = True
RUN_FULL_AFTER_TEST = False

TEST_CATE_1_DEPTH = "07. 스마트 사용성"
TEST_CATE_2_DEPTH = "07-06. 리모컨 사용성"

TEST_MAX_GROUPS = 1
MAX_GROUP_SAMPLE_MEMOS = 30
MAX_AUTO_TOPICS_PER_GROUP = 8
MAX_RETRIES = 3
RATE_LIMIT_SECONDS = 0.4
BACKOFF_BASE = 1.8
MAX_TOKENS = 1400

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"
AUTO_CATALOG_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_auto_topic_catalog_{PROMPT_VERSION}"
AUTO_DETAIL_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_auto_topic_detail_{PROMPT_VERSION}"
TAXONOMY_MAP_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_auto_topic_taxonomy_map_{PROMPT_VERSION}"
FINAL_DETAIL_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_topic_detail_{PROMPT_VERSION}"
FINAL_SUMMARY_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_topic_summary_{PROMPT_VERSION}"
FAILED_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_topic_failed_{PROMPT_VERSION}"

print(
    f"[CONFIG] quarter={SNAPSHOT_QUARTER}, prompt_version={PROMPT_VERSION}, "
    f"endpoint={ENDPOINT}, run_scope={RUN_SCOPE}"
)


# =========================
# Helpers
# =========================

def clean_text(text: Any) -> str:
    if text is None:
        return ""
    return str(text).replace("\n", " ").replace("\r", " ").strip()


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def normalize_sentiment(raw: Any) -> str:
    text = clean_text(raw)
    if text in {"긍정", "positive", "pos"}:
        return "긍정"
    if text in {"부정", "negative", "neg"}:
        return "부정"
    if text in {"중립", "중립/질문", "neutral", "question", "neutral/question"}:
        return "중립"
    return "기타"


def extract_json_object(text: str) -> Dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        return json.loads(match.group(0))
    raise ValueError(f"Invalid JSON from model: {text[:500]}")


def call_endpoint(messages: List[Dict[str, str]]) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client

    client = get_deploy_client("databricks")
    payload = {
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": MAX_TOKENS,
    }

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)
            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json_object(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json_object(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json_object(pred0)
                if "content" in resp:
                    return extract_json_object(resp["content"])
            if isinstance(resp, str):
                return extract_json_object(resp)
            raise ValueError(f"Unexpected response schema: {type(resp)}")
        except Exception as exc:
            last_error = exc
            time.sleep(BACKOFF_BASE ** attempt)

    raise RuntimeError(f"Endpoint request failed after retries: {last_error}")


def save_table(df: DataFrame, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] overwrite -> {table_name}")
        df.write.mode("overwrite").format("delta").saveAsTable(table_name)
    else:
        print(f"[SAVE] create -> {table_name}")
        df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def load_source_df(test_mode: bool = False) -> DataFrame:
    df = spark.sql(
        f"""
        SELECT DISTINCT cate_1_depth, cate_2_depth, sentiment, memo
        FROM {SOURCE_TABLE}
        WHERE memo IS NOT NULL
        """
    )
    if test_mode:
        df = (
            df.where(F.col("cate_1_depth") == F.lit(TEST_CATE_1_DEPTH))
              .where(F.col("cate_2_depth") == F.lit(TEST_CATE_2_DEPTH))
              .orderBy("sentiment", "memo")
              .limit(TEST_MAX_GROUPS * 100)
        )
    return df.withColumn("_row_id", F.monotonically_increasing_id())


def sample_group_memos_df(source_df: DataFrame) -> DataFrame:
    w = Window.partitionBy("cate_1_depth", "cate_2_depth", "sentiment").orderBy(F.length("memo").desc())
    sampled = (
        source_df
        .withColumn("_rn", F.row_number().over(w))
        .where(F.col("_rn") <= F.lit(MAX_GROUP_SAMPLE_MEMOS))
        .drop("_rn")
    )

    return (
        sampled.groupBy("cate_1_depth", "cate_2_depth", "sentiment")
        .agg(
            F.count("*").alias("row_cnt"),
            F.collect_list("memo").alias("sample_memos"),
        )
        .orderBy("cate_1_depth", "cate_2_depth", "sentiment")
    )


def stage1_extract_auto_catalog(group_row: Dict[str, Any]) -> List[Dict[str, Any]]:
    sample_memos = group_row["sample_memos"][:MAX_GROUP_SAMPLE_MEMOS]
    cate_1_depth = group_row["cate_1_depth"]
    cate_2_depth = group_row["cate_2_depth"]
    sentiment = group_row["sentiment"]

    system = (
        "You are a multilingual VOC analyst for TV reviews. "
        "Given a group of memo samples from one category/sentiment, extract an auto-topic catalog. "
        "Return only valid JSON with keys: topics. "
        "Each topic must have: auto_topic_id, auto_topic_label, keywords, coverage_hint. "
        "Rules: "
        "1) Produce 5 to 8 topics if possible. "
        "2) auto_topic_label should be Korean, sentence-like, and end with a natural statement style such as 함/있음/됨 when appropriate. "
        "3) Merge semantically similar expressions into one topic. "
        "4) Do not repeat near-duplicate topics. "
        "5) keywords should be short search terms that help assign rows later. "
        "6) Do not hardcode business taxonomy here. This is only the emergent auto-topic layer."
    )
    user = (
        f"cate_1_depth: {cate_1_depth}\n"
        f"cate_2_depth: {cate_2_depth}\n"
        f"sentiment: {sentiment}\n"
        f"sample_memos:\n- " + "\n- ".join(sample_memos) + "\n\n"
        f"Return a topic list no longer than {MAX_AUTO_TOPICS_PER_GROUP} items."
    )
    result = call_endpoint(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
    )
    topics = result.get("topics", [])
    if not isinstance(topics, list):
        topics = []

    cleaned: List[Dict[str, Any]] = []
    for idx, topic in enumerate(topics[:MAX_AUTO_TOPICS_PER_GROUP], start=1):
        if not isinstance(topic, dict):
            continue
        cleaned.append(
            {
                "cate_1_depth": cate_1_depth,
                "cate_2_depth": cate_2_depth,
                "sentiment": sentiment,
                "row_cnt": int(group_row["row_cnt"]),
                "auto_topic_id": clean_text(topic.get("auto_topic_id", f"A{idx:02d}")),
                "auto_topic_label": clean_text(topic.get("auto_topic_label", "기타 개선이 필요함")),
                "keywords": json.dumps(topic.get("keywords", []), ensure_ascii=False),
                "coverage_hint": clean_text(topic.get("coverage_hint", "")),
                "sample_memos": json.dumps(sample_memos, ensure_ascii=False),
            }
        )
    if not cleaned:
        cleaned = [
            {
                "cate_1_depth": cate_1_depth,
                "cate_2_depth": cate_2_depth,
                "sentiment": sentiment,
                "row_cnt": int(group_row["row_cnt"]),
                "auto_topic_id": "A01",
                "auto_topic_label": "기타 개선이 필요함",
                "keywords": json.dumps(["기타"], ensure_ascii=False),
                "coverage_hint": "fallback",
                "sample_memos": json.dumps(sample_memos, ensure_ascii=False),
            }
        ]
    return cleaned


def build_auto_catalog_df(sample_groups_df: DataFrame) -> DataFrame:
    rows = []
    for row in sample_groups_df.toLocalIterator():
        rows.extend(stage1_extract_auto_catalog(row.asDict(recursive=True)))
    return spark.createDataFrame(pd.DataFrame(rows))


def build_catalog_lookup(catalog_df: DataFrame) -> Dict[Tuple[str, str, str], List[Dict[str, Any]]]:
    lookup: Dict[Tuple[str, str, str], List[Dict[str, Any]]] = {}
    for row in catalog_df.toLocalIterator():
        key = (row["cate_1_depth"], row["cate_2_depth"], row["sentiment"])
        lookup.setdefault(key, []).append(
            {
                "auto_topic_id": row["auto_topic_id"],
                "auto_topic_label": row["auto_topic_label"],
                "keywords": json.loads(row["keywords"]) if row["keywords"] else [],
                "coverage_hint": row["coverage_hint"],
            }
        )
    return lookup


def normalize_label(text: str) -> str:
    text = normalize_whitespace(clean_text(text))
    return re.sub(r"\s+", " ", text)


def score_auto_topic(memo: str, topic: Dict[str, Any]) -> int:
    haystack = normalize_whitespace(clean_text(memo).lower())
    score = 0
    label = normalize_whitespace(str(topic.get("auto_topic_label", "")).lower())
    if label and label in haystack:
        score += 3
    for kw in topic.get("keywords", []):
        kw_text = normalize_whitespace(str(kw).lower())
        if kw_text and kw_text in haystack:
            score += 1
    return score


def build_auto_assignment_partition(catalog_lookup: Dict[Tuple[str, str, str], List[Dict[str, Any]]]):
    def partition(pdf_iter: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
        for pdf in pdf_iter:
            out_rows: List[Dict[str, Any]] = []
            for row in pdf.itertuples(index=False, name=None):
                cate_1_depth, cate_2_depth, sentiment, memo, row_id = row
                key = (cate_1_depth, cate_2_depth, sentiment)
                candidates = catalog_lookup.get(key, [])
                if not candidates:
                    candidates = [{"auto_topic_id": "A00", "auto_topic_label": "기타 개선이 필요함", "keywords": []}]
                scored = sorted(
                    candidates,
                    key=lambda t: (score_auto_topic(memo, t), len(t.get("keywords", []))),
                    reverse=True,
                )
                chosen = scored[0]
                out_rows.append(
                    {
                        "_row_id": row_id,
                        "cate_1_depth": cate_1_depth,
                        "cate_2_depth": cate_2_depth,
                        "sentiment": sentiment,
                        "memo": memo,
                        "auto_topic_id": chosen["auto_topic_id"],
                        "auto_topic_label": chosen["auto_topic_label"],
                        "auto_topic_score": score_auto_topic(memo, chosen),
                        "assignment_method": "keyword_match" if key in catalog_lookup else "fallback",
                    }
                )
            yield pd.DataFrame(out_rows)

    return partition


def assign_auto_topics_df(source_df: DataFrame, catalog_lookup: Dict[Tuple[str, str, str], List[Dict[str, Any]]]) -> DataFrame:
    schema = T.StructType(
        [
            T.StructField("_row_id", T.LongType(), True),
            T.StructField("cate_1_depth", T.StringType(), True),
            T.StructField("cate_2_depth", T.StringType(), True),
            T.StructField("sentiment", T.StringType(), True),
            T.StructField("memo", T.StringType(), True),
            T.StructField("auto_topic_id", T.StringType(), True),
            T.StructField("auto_topic_label", T.StringType(), True),
            T.StructField("auto_topic_score", T.IntegerType(), True),
            T.StructField("assignment_method", T.StringType(), True),
        ]
    )
    return source_df.mapInPandas(build_auto_assignment_partition(catalog_lookup), schema=schema)


def stage3_map_auto_topic_to_taxonomy(auto_topic_df: DataFrame) -> DataFrame:
    unique_topics = auto_topic_df.select(
        "cate_1_depth",
        "cate_2_depth",
        "sentiment",
        "auto_topic_id",
        "auto_topic_label",
    ).distinct()

    mappings: List[Dict[str, Any]] = []
    for row in unique_topics.toLocalIterator():
        system = (
            "You are a taxonomy mapper for TV VOC topics. "
            "Map the auto topic to a stable business taxonomy. "
            "Return only valid JSON with keys: taxonomy_lv1, taxonomy_lv2, taxonomy_label. "
            "Rules: "
            "1) taxonomy_lv1 should be a concise Korean family label. "
            "2) taxonomy_lv2 should be a concise Korean detailed label. "
            "3) Preserve the meaning of the auto topic. "
            "4) Do not merge unrelated meanings."
        )
        user = (
            f"cate_1_depth: {row['cate_1_depth']}\n"
            f"cate_2_depth: {row['cate_2_depth']}\n"
            f"sentiment: {row['sentiment']}\n"
            f"auto_topic_label: {row['auto_topic_label']}\n"
            f"auto_topic_id: {row['auto_topic_id']}\n\n"
            "Output JSON only."
        )
        result = call_endpoint(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ]
        )
        mappings.append(
            {
                "cate_1_depth": row["cate_1_depth"],
                "cate_2_depth": row["cate_2_depth"],
                "sentiment": row["sentiment"],
                "auto_topic_id": row["auto_topic_id"],
                "auto_topic_label": row["auto_topic_label"],
                "taxonomy_lv1": clean_text(result.get("taxonomy_lv1", "")) or row["cate_1_depth"],
                "taxonomy_lv2": clean_text(result.get("taxonomy_lv2", "")) or row["auto_topic_label"],
                "taxonomy_label": clean_text(result.get("taxonomy_label", "")) or f"{row['cate_1_depth']} / {row['auto_topic_label']}",
            }
        )
        time.sleep(RATE_LIMIT_SECONDS)

    return spark.createDataFrame(pd.DataFrame(mappings))


def build_final_detail_df(auto_detail_df: DataFrame, taxonomy_map_df: DataFrame) -> DataFrame:
    joined = auto_detail_df.join(
        F.broadcast(taxonomy_map_df),
        on=["cate_1_depth", "cate_2_depth", "sentiment", "auto_topic_id", "auto_topic_label"],
        how="left",
    )
    return joined.select(
        "_row_id",
        "cate_1_depth",
        "cate_2_depth",
        "sentiment",
        "memo",
        "auto_topic_id",
        "auto_topic_label",
        "auto_topic_score",
        "assignment_method",
        "taxonomy_lv1",
        "taxonomy_lv2",
        "taxonomy_label",
    )


def build_summary_df(detail_df: DataFrame) -> DataFrame:
    return (
        detail_df.groupBy("cate_1_depth", "cate_2_depth", "sentiment", "taxonomy_lv1", "taxonomy_lv2", "taxonomy_label")
        .agg(
            F.count("*").alias("row_cnt"),
            F.round((F.count("*") / F.sum(F.count("*")).over(Window.partitionBy("cate_1_depth", "cate_2_depth"))), 4).alias("row_share"),
        )
    )


def run_pipeline(test_mode: bool) -> Dict[str, DataFrame]:
    source_df = load_source_df(test_mode=test_mode)
    sample_groups_df = sample_group_memos_df(source_df)
    catalog_df = build_auto_catalog_df(sample_groups_df)
    catalog_lookup = build_catalog_lookup(catalog_df)
    auto_detail_df = assign_auto_topics_df(source_df, catalog_lookup)
    taxonomy_map_df = stage3_map_auto_topic_to_taxonomy(auto_detail_df)
    final_detail_df = build_final_detail_df(auto_detail_df, taxonomy_map_df)
    final_summary_df = (
        final_detail_df.groupBy("cate_1_depth", "cate_2_depth", "sentiment", "taxonomy_lv1", "taxonomy_lv2", "taxonomy_label")
        .agg(
            F.count("*").alias("row_cnt"),
            F.avg("auto_topic_score").alias("avg_auto_topic_score"),
        )
        .withColumn(
            "category_total_cnt",
            F.sum("row_cnt").over(Window.partitionBy("cate_1_depth", "cate_2_depth")),
        )
        .withColumn("row_share", F.round(F.col("row_cnt") / F.col("category_total_cnt"), 4))
        .withColumn("row_share_pct", F.round(F.col("row_share") * 100, 2))
    )

    save_table(catalog_df, AUTO_CATALOG_TABLE if not test_mode else f"{AUTO_CATALOG_TABLE}_test")
    save_table(auto_detail_df, AUTO_DETAIL_TABLE if not test_mode else f"{AUTO_DETAIL_TABLE}_test")
    save_table(taxonomy_map_df, TAXONOMY_MAP_TABLE if not test_mode else f"{TAXONOMY_MAP_TABLE}_test")
    save_table(final_detail_df, FINAL_DETAIL_TABLE if not test_mode else f"{FINAL_DETAIL_TABLE}_test")
    save_table(final_summary_df, FINAL_SUMMARY_TABLE if not test_mode else f"{FINAL_SUMMARY_TABLE}_test")

    display(catalog_df.toPandas())
    display(final_summary_df.toPandas())

    return {
        "catalog_df": catalog_df,
        "auto_detail_df": auto_detail_df,
        "taxonomy_map_df": taxonomy_map_df,
        "final_detail_df": final_detail_df,
        "final_summary_df": final_summary_df,
    }


# =========================
# Run
# =========================

test_result = run_pipeline(test_mode=True)

if RUN_FULL_AFTER_TEST:
    full_result = run_pipeline(test_mode=False)

